We want to demonstrate that it is possible to fine-tune a model that is not very capable to complete a fairly challenging task and do quite well. We are going to fine-tune Qwen2.5-Coder-1.5B to take textual descriptions of math equations and generate python code for the Mamim library that renders those expressions.

In the process, we will also use a hyperparameter optimization library called optuna that will search for the optimal hyperparameter values within our specified search space.

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 32.6 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
dataset = load_dataset("Edoh/manim_python")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/599 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/51 [00:00<?, ? examples/s]

In [3]:
# Let's look at an example from the training set
#
import random
r = random.randint(0, len(dataset['train'])-1)
print(dataset['train'][r])

{'instruction': 'Move the hexagon to the left by 2 units over 1 second.', 'output': 'from manim import * class MyScene(Scene): def construct(self): hexagon = RegularPolygon(n=6, radius=2, color=YELLOW) self.add(hexagon) self.play(hexagon.animate.shift(LEFT * 2), run_time=1)'}


In [30]:
# We need to load the model and its corresponding tokenizer. We will use the Qwen2.5-Coder-1.5B model for this example.
#
from transformers import AutoTokenizer
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add a new pad token that is different from the eos token
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})
else:
    print(f"Tokenizer already has a pad token: {tokenizer.pad_token}")

Tokenizer already has a pad token: <|endoftext|>


We need to prepare the dataset and put it into the format that the model expects it to be in for instruction fine-tuning.
When training a model for Causal Language Modeling (CL), the objective is to predict the next token in a sequence given the previous tokens.

In the Hugging Face transformers implementation (specifically for models like GPT2LMHeadModel), this "shifting" is handled internally by the model during the loss computation.

By setting labels equal to input_ids:

You provide the full ground-truth sequence as the target.
The model internally calculates the loss by comparing its prediction at position i with the label at position i. However, because the model is causal (it can only see tokens 0 to i), this effectively teaches it to predict the next token.

In [31]:
def preprocess_data(examples):
    inputs = [
        f"Instruction: {instr}\nOutput: {out}"
        for instr, out in zip(examples["instruction"], examples["output"])
    ]
    tokenized = tokenizer(inputs, truncation=True, max_length=512, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_datasets = dataset.map(preprocess_data,batched=True,
                                 remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/599 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

In [32]:
# Print out an example
print(tokenized_datasets['train'][0])

{'input_ids': [16664, 25, 4230, 264, 501, 6109, 6941, 364, 5050, 10019, 23569, 5097, 25, 504, 883, 318, 1159, 353, 536, 3017, 10019, 96116, 1648, 707, 9245, 1193, 1648, 1494, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 1


Optuna is an open-source hyperparameter optimization framework designed to automate the search for optimal hyperparameters in machine learning models. It is framework-agnostic and can be used with PyTorch, TensorFlow, Keras, Scikit-learn, XGBoost, LightGBM, and others.

Here is a summary of its key features:

1. Define-by-Run Style
Unlike traditional frameworks that require a static configuration file, Optuna allows you to construct the search space dynamically using Python code (e.g., loops, conditionals). This makes it highly flexible for complex parameter relationships.

2. Efficient Sampling Algorithms
Optuna uses state-of-the-art algorithms to sample hyperparameters intelligently, often finding better results faster than grid or random search:

TPE (Tree-structured Parzen Estimator): A Bayesian optimization approach (default).
CMA-ES (Covariance Matrix Adaptation Evolution Strategy): Good for continuous hyperparameters.
3. Pruning (Early Stopping)
Optuna can automatically stop unpromising trials early based on intermediate results (e.g., validation loss). This saves computational resources and time.

Median Pruner: Prunes if the trial's intermediate result is worse than the median of previous trials at the same step.
Hyperband Pruner: A bandit-based approach for resource allocation.
4. Easy Integration
It requires minimal code changes. You simply define an objective function that takes a trial object and returns a metric to optimize (like accuracy or loss).

In [8]:
# Define an initializer for the model since we will need to construct it multiple times
# during training and hyperparameter tunning.
#
from transformers import AutoModelForCausalLM

def model_init():
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')
    # Resize model embeddings to account for new pad token
    if tokenizer.pad_token_id is not None and model.config.vocab_size < len(tokenizer):
        model.resize_token_embeddings(len(tokenizer))
    return model

In [15]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./Qwen2.5-Coder-1.5B-manim-python-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False, # Changed to False to fix the BFloat16 error
    report_to="none",
)

In [35]:
# We need some data to use for evaluation during training. We can create a
# validation set by splitting the training set.
#
train_val_split = tokenized_datasets["train"].train_test_split(test_size=0.1)
tokenized_datasets["train"] = train_val_split["train"]
tokenized_datasets["validation"] = train_val_split["test"]

In [36]:
from transformers import (
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(model_init=model_init,
                  args=training_args,
                  train_dataset=tokenized_datasets["train"],
                  eval_dataset=tokenized_datasets["validation"],
                  processing_class=tokenizer,
                  data_collator=data_collator,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [18]:
# These methods are the core mechanism Optuna uses to sample hyperparameters from a defined
# search space. When you call them, Optuna picks a value for that specific trial based on the
# history of previous trials (using its internal sampling algorithm, typically TPE).
#
def hp_space(trial):
    return {
    "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
    "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size",  [2, 4, 8]),
    "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
    "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 6),
    "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
    "gradient_accumulation_steps": trial.suggest_categorical("gradient_accumulation_steps", [1, 2, 4])
    }

In [19]:
best_run = trainer.hyperparameter_search(
    direction="minimize",
    backend="optuna",
    n_trials=10,
    hp_space=hp_space,
    compute_objective=lambda metrics: metrics["eval_loss"],
)

[I 2026-02-26 14:29:52,967] A new study created in memory with name: no-name-bbebe5d0-c72b-4bc0-af1c-b845587ffcdf
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.300930
2,No log,0.165209
3,No log,0.165701
4,No log,0.174121


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss
1,No log,0.743574


[I 2026-02-26 14:48:43,877] Trial 8 pruned. 
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,0.584745


[I 2026-02-26 14:49:07,373] Trial 9 pruned. 


In [20]:
# Let's see what values it came up with
#
for key, value in best_run.hyperparameters.items():
    print(f"{key}: {value}")

learning_rate: 1.814411983038922e-05
per_device_train_batch_size: 2
weight_decay: 0.078225967580963
num_train_epochs: 4
warmup_steps: 152
gradient_accumulation_steps: 2


In [37]:
# We can now pick the best hyperparameters and train a final model using those hyperparameters.
#
for key, value in best_run.hyperparameters.items():
    setattr(training_args, key, value)

trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets.get("validation"),
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [38]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,1.844322,0.026622
2,0.021525,0.019720
3,0.014954,0.018918
4,0.014005,0.018626


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=540, training_loss=0.35533324733928395, metrics={'train_runtime': 252.0405, 'train_samples_per_second': 8.554, 'train_steps_per_second': 2.143, 'total_flos': 8678689845805056.0, 'train_loss': 0.35533324733928395, 'epoch': 4.0})

In [39]:
trainer.save_model("./Qwen2.5-Coder-1.5B-manim-python-finetuned")
tokenizer.save_pretrained("./Qwen2.5-Coder-1.5B-manim-python-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./Qwen2.5-Coder-1.5B-manim-python-finetuned/tokenizer_config.json',
 './Qwen2.5-Coder-1.5B-manim-python-finetuned/chat_template.jinja',
 './Qwen2.5-Coder-1.5B-manim-python-finetuned/tokenizer.json')

In [40]:
import torch
import tiktoken
from transformers import AutoTokenizer, AutoModelForCausalLM

model_dir = "./Qwen2.5-Coder-1.5B-manim-python-finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForCausalLM.from_pretrained(model_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Loading model onto device ' + str(device))
model.to(device)
model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading model onto device cuda


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [41]:
def generate_output(instruction, max_length=150, num_beams=5,
                    temperature=0.7, top_p=0.9, repetition_penalty=1.2):
  prompt = f"Instruction: {instruction}\nOutput:"
  input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
  generated_ids = model.generate(
    input_ids,
    max_length=max_length,
    num_beams=num_beams,
    temperature=temperature,
    top_p=top_p,
    repetition_penalty=repetition_penalty,
    do_sample=True,
    pad_token_id=tokenizer.pad_token_id, # Use the new pad_token_id
    eos_token_id=tokenizer.eos_token_id,
    early_stopping=True,
    no_repeat_ngram_size=2,
  )
  generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
  output_start = generated_text.find("Output:")
  if output_start != -1:
    output_text = generated_text[output_start + len("Output:"):].strip()
  else:
    output_text = generated_text.strip()
  return output_text

In [42]:
import csv
output_csv = "Qwen2.5-Coder_manim_python_test_outputs.csv"
with open(output_csv, mode="w", newline="", encoding="utf-8") as csvfile:
  writer = csv.DictWriter(csvfile,
  fieldnames=["instruction", "reference_output", "generated_output"])
  writer.writeheader()

  for example in dataset['test']:
    instruction = example["instruction"]
    reference_output = example["output"]
    generated_output = generate_output(instruction)
    writer.writerow({
      "instruction": instruction,
      "reference_output": reference_output,
      "generated_output": generated_output
    })

In [ ]:
import ast

def is_syntax_valid(code_str):
  try:
    ast.parse(code_str)
    return True, ""
  except SyntaxError as e:
    return False, str(e)

class ManimCodeAnalyzer(ast.NodeVisitor):
  def __init__(self):
    self.imports_manim = False
    self.scene_subclass_names = []
    self.play_calls = 0
    self.create_calls = 0
    self.errors = []

  def visit_Import(self, node):
    for alias in node.names:
      if alias.name == "manim":
        self.imports_manim = True
    self.generic_visit(node)

  def visit_ImportFrom(self, node):
    if node.module and node.module.startswith("manim"):
      self.imports_manim = True
    self.generic_visit(node)

  def visit_ClassDef(self, node):
    for base in node.bases:
      if isinstance(base, ast.Name) and base.id == "Scene":
        self.scene_subclass_names.append(node.name)
      elif isinstance(base, ast.Attribute):
        if base.attr == "Scene":
          self.scene_subclass_names.append(node.name)
    self.generic_visit(node)

  def visit_Call(self, node):
    if isinstance(node.func, ast.Attribute):
      if (isinstance(node.func.value, ast.Name) and node.func.value.id == "self" and node.func.attr == "play"):
        self.play_calls += 1
      if isinstance(node.func, ast.Name) and node.func.id == "Create":
        self.create_calls += 1
    self.generic_visit(node)

def analyze_manim_code(code_str):
  analyzer = ManimCodeAnalyzer()
  try:
    tree = ast.parse(code_str)
  except SyntaxError as e:
    return {
      "syntax_valid": False,
      "syntax_error": str(e),
      "imports_manim": False,
      "scene_subclass_names": [],
      "play_calls": 0,
      "create_calls": 0,
      }
  analyzer.visit(tree)
  return {
    "syntax_valid": True,
    "syntax_error": None,
    "imports_manim": analyzer.imports_manim,
    "scene_subclass_names": analyzer.scene_subclass_names,
    "play_calls": analyzer.play_calls,
    "create_calls": analyzer.create_calls,
  }

In [ ]:
# Test it out using some manim code
#
manim_code_sample = """
from manim import *

class MyScene(Scene):
  def construct(self):
    circle = Circle()
    self.play(Create(circle))
"""

result = analyze_manim_code(manim_code_sample)
print(result)


{'syntax_valid': True, 'syntax_error': None, 'imports_manim': True, 'scene_subclass_names': ['MyScene'], 'play_calls': 1, 'create_calls': 0}


In [ ]:
import os
import subprocess
import tempfile

def evaluate_manim_code(code_str, scene_class_name="CustomScene"):
  with tempfile.TemporaryDirectory() as tmpdir:
    code_path = os.path.join(tmpdir, "generated_scene.py")
    with open(code_path, "w") as f:
      f.write(code_str)

  cmd = [
    "manim",
    "-ql",
    code_path,
    scene_class_name,
  ]
  try:
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    success = result.returncode == 0
    output = result.stdout + "\n" + result.stderr
  except subprocess.TimeoutExpired:
    success = False
    output = "Timeout expired during rendering."
  return success, output


In [ ]:
# Test the evaluate_manim_code function
#
success, output = evaluate_manim_code(manim_code_sample, scene_class_name="MyScene")
print("Render success:", success)
print("Output:", output)

FileNotFoundError: [Errno 2] No such file or directory: 'manim'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install manim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 72.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
